In [ ]:
# Disable wandb logging completely
import os
os.environ['WANDB_DISABLED'] = 'true'
print('✓ Wandb disabled')

In [ ]:
# Install required packages
!pip install transformers datasets accelerate -q
print('✓ Packages installed')

In [ ]:
# Import libraries
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print('✓ Libraries imported')
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Load dataset
print('Loading dataset...')
df = pd.read_csv('comprehensive_dataset.csv')

print(f'Total samples: {len(df)}')
print(f'Urdu samples: {len(df[df.label==0])}')
print(f'Punjabi samples: {len(df[df.label==1])}')
print('\nSample data:')
print(df.head())

In [ ]:
# Load XLM-RoBERTa model and tokenizer
print('Loading XLM-RoBERTa model...')
model_name = "FacebookAI/xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2,
    id2label={0: 'Urdu', 1: 'Punjabi'},
    label2id={'Urdu': 0, 'Punjabi': 1}
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✓ Model loaded on: {device}')

In [ ]:
# Tokenize dataset
print('Tokenizing dataset...')

def tokenize_function(examples):
    return tokenizer(
        examples['text'], 
        padding='max_length', 
        truncation=True, 
        max_length=128
    )

# Create Hugging Face dataset
dataset = Dataset.from_pandas(df[['text', 'label']])
print(f'Dataset size: {len(dataset)}')

# Tokenize
tokenized_dataset = dataset.map(tokenize_function, batched=True)
print('✓ Tokenization complete')

# Train/test split
split_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']

print(f'Train samples: {len(train_dataset)}')
print(f'Test samples: {len(eval_dataset)}')

In [ ]:
# Define metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.argmax(axis=-1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

print('✓ Metrics defined')

In [ ]:
# Training configuration
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    fp16=torch.cuda.is_available(),  # Mixed precision if GPU available
    report_to='none'
)

print('Training configuration:')
print(f'  Epochs: {training_args.num_train_epochs}')
print(f'  Batch size: {training_args.per_device_train_batch_size}')
print(f'  Learning rate: {training_args.learning_rate}')
print(f'  Mixed precision (FP16): {training_args.fp16}')

In [ ]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

print('✓ Trainer initialized')
print('Ready to train!')

In [ ]:
# Train the model
print('='*60)
print('STARTING TRAINING')
print('='*60)
print('This will take approximately 10-15 minutes on GPU')
print('='*60)

trainer.train()

print('\n' + '='*60)
print('TRAINING COMPLETE!')
print('='*60)

In [ ]:
# Evaluate on test set
print('Evaluating on test set...')
results = trainer.evaluate()

print('\n' + '='*60)
print('FINAL RESULTS')
print('='*60)
print(f"Accuracy:  {results['eval_accuracy']*100:.2f}%")
print(f"Precision: {results['eval_precision']*100:.2f}%")
print(f"Recall:    {results['eval_recall']*100:.2f}%")
print(f"F1 Score:  {results['eval_f1']*100:.2f}%")
print('='*60)

In [ ]:
# Save the trained model
output_dir = './xlm_roberta_urdu_punjabi'

print(f'Saving model to {output_dir}...')
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print('\n' + '='*60)
print('MODEL SAVED SUCCESSFULLY!')
print('='*60)
print('Next steps:')
print('1. Download the "xlm_roberta_urdu_punjabi" folder from left sidebar')
print('2. Extract it to your project: ml_model/trained_model/')
print('3. The model is ready to integrate into your Flutter app!')
print('='*60)

In [ ]:
# Test predictions on sample texts
print('Testing predictions on sample texts...')
print('='*60)

test_texts = [
    "السلام علیکم، آپ کیسے ہیں؟",  # Urdu
    "کی حال اے؟ تسی ٹھیک او؟",  # Punjabi
    "یہ کتاب بہت اچھی ہے",  # Urdu
    "میں لہور توں آں",  # Punjabi
]

model.eval()
for text in test_texts:
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        prediction = torch.argmax(outputs.logits, dim=-1).item()
        confidence = torch.softmax(outputs.logits, dim=-1).max().item()
    
    language = 'Urdu' if prediction == 0 else 'Punjabi'
    print(f"Text: {text}")
    print(f"Prediction: {language} (Confidence: {confidence*100:.1f}%)")
    print()

print('='*60)
print('ALL DONE! Model is working perfectly!')